In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery, QueryType, QueryCaptionType, QueryAnswerType
from config import AZURE_AI_SEARCH_ENDPOINT, AZURE_AI_SEARCH_KEY, AZURE_AI_SEARCH_RFP_INDEX_NAME, AZURE_AI_SEARCH_RFI_INDEX_NAME
import os



AZURE_AI_SEARCH_ENDPOINT=os.getenv("AZURE_AI_SEARCH_ENDPOINT")
AZURE_AI_SEARCH_KEY = os.getenv("AZURE_AI_SEARCH_KEY")
AZURE_AI_SEARCH_RFI_INDEX_NAME= os.getenv("AZURE_AI_SEARCH_RFI_INDEX_NAME")
AZURE_AI_SEARCH_RFP_INDEX_NAME=os.getenv("AZURE_AI_SEARCH_RFP_INDEX_NAME")
field_name = "project_id" 

# Initialize clients
credential = AzureKeyCredential(AZURE_AI_SEARCH_KEY)
client_a = SearchClient(endpoint=AZURE_AI_SEARCH_ENDPOINT, index_name=AZURE_AI_SEARCH_RFI_INDEX_NAME, credential=credential)
client_b = SearchClient(endpoint=AZURE_AI_SEARCH_ENDPOINT, index_name=AZURE_AI_SEARCH_RFP_INDEX_NAME, credential=credential)

query_1="what is the scope of work for merivale?"
query_2="what are the required activities for merivale?"
query_text="what is the scope of work?"

# 1. Query Index A to get top 50 chunks
vector_query_1 = VectorizableTextQuery(
        text=query_1,
        k_nearest_neighbors=50,
        fields="scope_of_work_vectorized",  # Must match your index
        exhaustive=True,  # Use exhaustive search for better accuracy
        weight=2
    )
vector_query_2 = VectorizableTextQuery(
        text=query_2,
        k_nearest_neighbors=50,
        fields="required_activities_vectorized",  # Must match your index
        exhaustive=True,  # Use exhaustive search for better accuracy
        weight=0.5
    )
response_a = client_a.search(
    search_text="*", 
    vector_queries=[vector_query_1, vector_query_2],  
    query_type=QueryType.SEMANTIC,
    semantic_configuration_name='my-semantic-config',
    query_language="en",
    query_caption=QueryCaptionType.EXTRACTIVE,
    vector_filter_mode="postFilter",             # Or your specific query
    top=10,
    select=[field_name],              # Only need this field
    include_total_count=False
)

# Extract the values of the specified field from results
values = set()
for doc in response_a:
    if field_name in doc:
        values.add(doc[field_name])
print(values)
# Print results
def print_results(results):
    for result in results:
        print(f"File: {result['file_name']}")
        print(f"Section: {result['section_name']}")
        print(f"Chunk ID: {result['chunk_id']}")
        print(f"Score: {result['@search.score']}")
        print(f"Content: {result['content']}")
        print(f"Section No: {result['section_no']}")
        print("-" * 40)
if not values:
    print(f"No values found for field '{field_name}' in index A results.")
else:
    # Build an OData filter string to match any of the retrieved values
    # Azure filters use: field eq 'value'
    # Use search.in for efficiency when matching multiple values
    import json
    safe_values = [v.replace("'", "''") for v in values]  # Escape single quotes
    # values_list = ",".join(f"'{v}'" for v in safe_values)
    values_list = f"'{','.join(safe_values)}'"
    print(values_list)
    filter_expr = f"search.in({field_name} , {values_list},  ',')"
    print(filter_expr)
    vector_query_3 = VectorizableTextQuery(
        text=query_text,
        k_nearest_neighbors=50,
        fields="content_vector",  # Must match your index
        query_rewrites="generative|count-5" ,    # or "generative" if your index supports it
        exhaustive=True,  # Use exhaustive search for better accuracy
    )

    # 2. Query Index B using the filter to get top 15 chunks
    response_b = client_b.search(
        search_text="What is the scope of work?",
        filter=filter_expr,
        query_type=QueryType.SEMANTIC,
        semantic_configuration_name='my-semantic-config',
        query_language="en",
        query_caption=QueryCaptionType.EXTRACTIVE,
        vector_queries=[vector_query_3],
        vector_filter_mode="postFilter",
        top=15,
        select=["chunk_id", "domain", "content_type", "content",'file_name',"section_name"]
    )
    print(response_b)
    print(f"Top 15 chunks from index B where {field_name} matches values from A:")
    for doc in response_b:
        print("Index A doc:", doc)


{'705-22318727.00'}
'705-22318727.00'
search.in(project_id , '705-22318727.00',  ',')
<iterator object azure.core.paging.ItemPaged at 0x1ba33fb05f0>
Top 15 chunks from index B where project_id matches values from A:
Index A doc: {'section_name': 'SCOPE OF WORK\n\nT', 'domain': 'Energy Infrastructure & Grid Modernization', 'content': '3.0 SCOPE OF WORK\n\nThe engineering service at Merivale TS includes the scope of work provided in the RFP documents under two (2) separate ARs. At a high level, this work consists of:\n\nAR 24127\n\n· Replace transformer T22 (New T24) and associated P&C/switchgear\n\n· Install 115kV P&C panels in the existing Control Building\n\n· Expand station fence line west\n\n 2\n\n markup_705-2231872700-PRO-G0001-00 Merivale TS - Sust and Develp.docx\n\n MERIVALE TS ENGINEERING SERVICES - SUSTAINMENT AND DEVELOPMENT AR24127 & AR23709 | JUNE 1, 2022 | 705-2231872700-PRO-G0001-00\n\n· Replace 4 oil breakers with SF6 breakers\n\n· Install 6 new SF6 breakers in 115kV ex